In [1]:
from pathlib import Path
import pandas as pd

In [4]:
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "../data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data directory: {DATA_DIR}")

Project root: /Users/tolulopesoetan/Desktop/pl_prediction_model/notebooks
Processed data directory: /Users/tolulopesoetan/Desktop/pl_prediction_model/notebooks/../data/processed


In [7]:
# load dataset
urls = {
    "2223": "https://www.football-data.co.uk/mmz4281/2223/E0.csv",
    "2324": "https://www.football-data.co.uk/mmz4281/2324/E0.csv",
    "2425": "https://www.football-data.co.uk/mmz4281/2425/E0.csv",
}

frames = []

for season, url in urls.items():
    df = pd.read_csv(url)
    df["season"] = season
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

print("Raw shape:", raw.shape)
print("\nColumns:")
print(raw.columns.tolist())

Raw shape: (1140, 133)

Columns:
['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'B365CH', 'B365CD', 'B365CA', 'BWCH', 'BWCD', 'BWCA', 'IWCH', 'IWCD', 'IWCA', 'PSCH', 'PSCD', 'PSCA', 'WHCH', 'WHCD', 'WHCA', 'VCCH', 'VCCD', 'VCCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'PC<2.5', 'MaxC>2.5', 'MaxC<2.5', 'AvgC>2.5', 'AvgC<2.5', 'AHCh', 'B365CAHH', 'B365CAHA', 'PCAHH', 'PCAHA', 'MaxCAHH', 'MaxCAHA', 'AvgCAHH', 'AvgCAHA', 'season', 'BFH', 'BFD', 'BFA', '1XBH', '1XBD'

In [8]:
raw[
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTR",
        "FTHG",
        "FTAG",
        "HTHG",
        "HTAG",
        "B365H",
        "B365D",
        "B365A",
    ]
].head(10)


,Date,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HTHG,HTAG,B365H,B365D,B365A
0,05/08/2022,Crystal Palace,Arsenal,A,0,2,0,1,4.20,3.60,1.85
1,06/08/2022,Fulham,Liverpool,D,2,2,1,0,11.00,6.00,1.25
2,06/08/2022,Bournemouth,Aston Villa,H,2,0,1,0,3.75,3.50,2.00
3,06/08/2022,Leeds,Wolves,H,2,1,1,1,2.25,3.40,3.20
4,06/08/2022,Newcastle,Nott'm Forest,H,2,0,0,0,1.66,3.80,5.25
5,06/08/2022,Tottenham,Southampton,H,4,1,2,1,1.33,5.50,8.50
6,06/08/2022,Everton,Chelsea,A,0,1,0,1,5.50,4.00,1.61
7,07/08/2022,Leicester,Brentford,D,2,2,1,0,2.00,3.75,3.50
8,07/08/2022,Man United,Brighton,A,1,2,0,2,1.60,4.20,5.25
9,07/08/2022,West Ham,Man City,A,0,2,0,1,8.00,5.00,1.36


In [9]:
# checking for missing values
cols_to_check = [
    "FTR",
    "FTHG",
    "FTAG",
    "HTHG",
    "HTAG",
    "B365H",
    "B365D",
    "B365A",
    "HS",
    "AS",
    "HST",
    "AST",
]

raw[cols_to_check].isnull().sum()

FTR      0
FTHG     0
FTAG     0
HTHG     0
HTAG     0
B365H    0
B365D    0
B365A    0
HS       0
AS       0
HST      0
AST      0
dtype: int64

In [10]:
print("Full-time result distribution:")
print(raw["FTR"].value_counts(dropna=False))

print("\nRows per season:")
print(raw.groupby("season")["FTR"].count())

Full-time result distribution:
FTR
H    514
A    364
D    262
Name: count, dtype: int64

Rows per season:
season
2223    380
2324    380
2425    380
Name: FTR, dtype: int64


In [ ]:
# clean rows and parse dates

raw["Date"] = pd.to_datetime(raw["Date"], dayfirst=True, errors="coerce")

required_cols = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "FTHG",
    "FTAG",
    "HTHG",
    "HTAG",
    "B365H",
    "B365D",
    "B365A",
]

raw = (
    raw.dropna(subset=required_cols)
       .sort_values("Date")
       .reset_index(drop=True)
       .copy()
)

print("Cleaned shape:", raw.shape)
print("Date range:", raw["Date"].min(), "to", raw["Date"].max())

Cleaned shape: (1140, 133)
Date range: 2022-08-05 00:00:00 to 2025-05-25 00:00:00


In [15]:
# build bookmaker market probabilities

raw["mkt_h"] = 1 / raw["B365H"]
raw["mkt_d"] = 1 / raw["B365D"]
raw["mkt_a"] = 1 / raw["B365A"]

margin = raw["mkt_h"] + raw["mkt_d"] + raw["mkt_a"]

raw["market_home_prob"] = raw["mkt_h"] / margin
raw["market_draw_prob"] = raw["mkt_d"] / margin
raw["market_away_prob"] = raw["mkt_a"] / margin

#raw

raw["market_prediction"] = (
    raw[["market_home_prob", "market_draw_prob", "market_away_prob"]]
    .idxmax(axis=1)
    .map(
        {
            "market_home_prob": "H",
            "market_draw_prob": "D",
            "market_away_prob": "A",
        }
    )
)

market_acc = (raw["market_prediction"] == raw["FTR"]).mean()

print(f"Market baseline accuracy: {market_acc:.3f}")
print("\nMarket prediction counts:")
print(raw["market_prediction"].value_counts())

Market baseline accuracy: 0.565

Market prediction counts:
market_prediction
H    722
A    418
Name: count, dtype: int64


In [16]:
raw[
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTR",
        "market_home_prob",
        "market_draw_prob",
        "market_away_prob",
        "market_prediction",
        "season",
    ]
].head(10)

,Date,HomeTeam,AwayTeam,FTR,market_home_prob,market_draw_prob,market_away_prob,market_prediction,season
0,2022-08-05,Crystal Palace,Arsenal,A,0.225381,0.262944,0.511675,A,2223
1,2022-08-06,Fulham,Liverpool,D,0.085960,0.157593,0.756447,A,2223
2,2022-08-06,Bournemouth,Aston Villa,H,0.253394,0.271493,0.475113,A,2223
3,2022-08-06,Leeds,Wolves,H,0.422853,0.279829,0.297318,H,2223
4,2022-08-06,Newcastle,Nott'm Forest,H,0.570440,0.249192,0.180368,H,2223
5,2022-08-06,Tottenham,Southampton,H,0.715160,0.172939,0.111901,H,2223
6,2022-08-06,Everton,Chelsea,A,0.172677,0.237431,0.589891,A,2223
7,2022-08-07,Leicester,Brentford,D,0.475113,0.253394,0.271493,H,2223
8,2022-08-07,Man United,Brighton,A,0.593220,0.225989,0.180791,H,2223
9,2022-08-07,West Ham,Man City,A,0.117892,0.188627,0.693481,A,2223


In [17]:
# save processed data

output_path = DATA_DIR / "raw_matches.csv"
raw.to_csv(output_path, index=False)

print(f"Saved cleaned matches to: {output_path}")

Saved cleaned matches to: /Users/tolulopesoetan/Desktop/pl_prediction_model/notebooks/../data/processed/raw_matches.csv
